In [1]:
import pandas as pd

import os
import glob
for dirname, _, filenames in os.walk('/spam-detection'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
excel_files = glob.glob("*.xlsx")
excel_files

['2024- text comments.xlsx.xlsx',
 'Cycle 1- text comments.xlsx.xlsx',
 'JAN 2025- text comments.xlsx.xlsx',
 'MAR 2025 Reporting- text comments.xlsx.xlsx']

In [3]:
data = pd.concat([pd.read_excel(file) for file in excel_files]).reset_index()

In [4]:
# Fill NaN values in 'comments' with empty string
data.fillna({"comments": ""}, inplace=True)

In [5]:
data

,index,respondent id,question code,comments,hide comment,language id,edited text,english translation,positive sentiment probability,neutral sentiment probability,negative sentiment probability,sentiment category
0,0,26298527,adt|action_vision,.,True,en,NaN,NaN,0.004242,0.989484,0.006274,neutral
1,1,26298527,ent|actions_accountability,.,True,en,NaN,NaN,0.004242,0.989484,0.006274,neutral
2,2,26298527,eet|actions_fearfrustration,.,True,en,NaN,NaN,0.004242,0.989484,0.006274,neutral
3,3,26298527,ctt|comments,"GRS has been struggling for years, and every y...",False,en,NaN,NaN,0.001122,0.002276,0.996603,negative
4,4,26298527,adt|action_communication,.,True,en,NaN,NaN,0.004242,0.989484,0.006274,neutral
...,...,...,...,...,...,...,...,...,...,...,...,...
2419,1189,26418802,dq|techs_comments,We still don't have a complete picture of wher...,False,en,NaN,NaN,0.000919,0.013322,0.985759,negative
2420,1190,26418482,dq|change_observed,I only joined in October 2024 therefore I can'...,True,en,NaN,NaN,NaN,NaN,NaN,NaN
2421,1191,26419987,dq|change_observed,,True,en,NaN,NaN,NaN,NaN,NaN,NaN
2422,1192,26419989,dq|change_observed,AI \nCopilot,False,en,NaN,NaN,0.004276,0.814515,0.181209,neutral


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
import re

In [7]:
def clean_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = text.lower().strip()
    return text

In [8]:
data.columns

Index(['index', 'respondent id', 'question code', 'comments', 'hide comment',
       'language id', 'edited text', 'english translation',
       'positive sentiment probability', 'neutral sentiment probability',
       'negative sentiment probability', 'sentiment category'],
      dtype='object')

In [9]:
# Fill NaN values in 'comments' with empty string
data.fillna({"comments": ""}, inplace=True)

In [10]:
# data['comments'] = data['comments'].apply(clean_text)
X = data[['comments', 'question code']]
y = data['hide comment']
X = pd.get_dummies(X, columns=['question code'])

In [11]:
# Vectorize 'comments' column using TF-IDF
vectorizer = TfidfVectorizer()
comments_vectorized = vectorizer.fit_transform(X['comments']).toarray()

In [12]:
X

,comments,question code_acn|comms_peers,question code_ada|info_awareness_presentation,question code_adb|agree_direction,question code_adb|conf_lv1_ldr,question code_adb|info_apollo_champs,question code_adb|info_apollo_core,question code_adb|info_managers,question code_adb|info_written,question code_adb|understand_purpose,...,question code_tra|issues_communication,question code_tra|issues_mgmt_support,question code_tra|issues_resistance,question code_trb|changing_size_shape,question code_trb|internal_restructure,question code_trb|new_way_of_working,question code_trb|overall_growth,question code_trb|pace,question code_trb|solving_business_challenges,question code_trb|solving_service_issues
0,.,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,.,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,.,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,"GRS has been struggling for years, and every y...",False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,.,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2419,We still don't have a complete picture of wher...,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2420,I only joined in October 2024 therefore I can'...,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2421,,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2422,AI \nCopilot,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [13]:
# Drop the original 'comments' column and add the TF-IDF features
X = X.drop(columns=['comments'])
X = pd.concat([X.reset_index(drop=True),
               pd.DataFrame(comments_vectorized, index=X.index)], axis=1)
X.columns = X.columns.astype(str)


In [14]:
# Fill any remaining NaN values in X with 0 before splitting
X.fillna(0, inplace=True)

In [15]:
# X.columns = X.columns.astype(str)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [17]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import MultinomialNB

models = [
    RandomForestClassifier(random_state=42),
    GradientBoostingClassifier(random_state=42),
    AdaBoostClassifier(random_state=42),
    ExtraTreesClassifier(random_state=42),
    BaggingClassifier(random_state=42),
    SVC(random_state=42),
    XGBClassifier(random_state=42),
    CatBoostClassifier(random_state=42, verbose=False),
    LGBMClassifier(random_state=42, verbose=-1),
    MultinomialNB()
]


In [18]:
for model in models:
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        print(model.__class__.__name__)
        print(classification_report(y_test, y_pred))
        print("Accuracy:", accuracy_score(y_test, y_pred))
        print("-" * 30)
    except Exception as e:
        print(model.__class__.__name__, "failed:", e)

RandomForestClassifier
              precision    recall  f1-score   support

       False       0.98      0.95      0.97       438
        True       0.88      0.96      0.92       168

    accuracy                           0.95       606
   macro avg       0.93      0.95      0.94       606
weighted avg       0.95      0.95      0.95       606

Accuracy: 0.9521452145214522
------------------------------
GradientBoostingClassifier
              precision    recall  f1-score   support

       False       0.98      0.93      0.96       438
        True       0.84      0.96      0.90       168

    accuracy                           0.94       606
   macro avg       0.91      0.94      0.93       606
weighted avg       0.94      0.94      0.94       606

Accuracy: 0.9389438943894389
------------------------------
AdaBoostClassifier
              precision    recall  f1-score   support

       False       0.89      0.93      0.91       438
        True       0.80      0.71      0.75     

In [15]:
model = SVC(random_state=42)

In [16]:
# model.fit(X_train_vect, y_train)
model.fit(X_train, y_train)

SVC(random_state=42)

In [17]:
y_pred = model.predict(X_test)

In [18]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       1.00      0.98      0.99       353
        True       0.94      0.99      0.97       132

    accuracy                           0.98       485
   macro avg       0.97      0.98      0.98       485
weighted avg       0.98      0.98      0.98       485



In [19]:
import joblib

joblib.dump(model, 'model.pkl')
joblib.dump(X.columns, 'model_features.pkl')

['model_features.pkl']

model = joblib.load('model.pkl')
model_features = joblib.load('model_features.pkl')

# Function to make predictions
def predict_hide_comment(comments, sentiment_category) -> bool:
    # Create a DataFrame for the input data
    input_data = pd.DataFrame({
        # 'question code': [question_code],
        'comments': [comments],
        'sentiment category': [sentiment_category],
    })
    
    # Convert categorical data to numerical data for 'question code'
    input_data = pd.get_dummies(input_data, columns=['sentiment category'])
    
    # Vectorize 'comments' column using the loaded TF-IDF vectorizer

    comments_tfidf = vectorizer.transform(input_data['comments']).toarray()
    
    # Drop the original 'comments' column and add the TF-IDF features
    input_data = input_data.drop(columns=['comments'])
    input_data = pd.concat([input_data, pd.DataFrame(comments_tfidf, index=input_data.index)], axis=1)

    # Reindex input_data to match the columns used during training
    # Fill any missing columns with 0
    input_data = input_data.reindex(columns=model_features, fill_value=0)
    
    input_data.columns = input_data.columns.astype(str)
    
    # Make prediction
    prediction = model.predict(input_data)
    
    return prediction[0]


# Example usage
# question_code = 'dq|change_observed'
comments = "There has been an introduction to AI solutions and tools that have helped with workflow."
sentiment_category = "neutral"

result = predict_hide_comment(comments, sentiment_category)
print(f"Prediction: {result}")